In [ ]:
!pip install -q openai

In [ ]:
import json
import os
import subprocess
import threading
import time
import uuid
import glob as glob_module
import shutil
import tempfile
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum, auto
from pathlib import Path
from typing import Callable, Optional
from openai import OpenAI
import tiktoken

os.environ['OPENAI_API_KEY'] = ""
client = OpenAI()
MODEL = 'gpt-5-mini'

# 1) Agent Loop

The agent loop is the **single architectural primitive** shared by every LLM agent.
It is deceptively simple:

1. Send `messages[]` + tool schemas to the LLM.
2. If `finish_reason == 'stop'` -- the model is done, return.
3. If `finish_reason == 'tool_calls'` -- execute each requested tool, append results as `role: 'tool'` messages, loop back to step 1.

The model decides *when* to call tools and *when* to stop.
The harness executes calls and maintains state.


```
Input:  messages = [{role: user, content: What Python version is installed?}]

Iter 1: LLM requests tool -> bash(command='python3 --version')
        Harness executes  -> 'Python 3.11.4'

Iter 2: LLM reads result  -> finish_reason = stop
Output: [..., {role: assistant, content: Python 3.11.4 is installed.}]
```

**OpenAI vs Anthropic Differences**

| Aspect | OpenAI | Anthropic |
|--------|--------|-----------|
| Stop signal | `finish_reason == 'tool_calls'` | `stop_reason == 'tool_use'` |
| Tool result role | `'tool'` + `tool_call_id` | `'user'` |
| Schema key | `{type: function, function: {...}}` | `{name: ..., input_schema: ...}` |

In [ ]:
def run_agent_loop(
    messages,
    tools,
    tool_handlers,
    system='',
    max_iterations=20,
    on_tool_call=None,
):
    """
    messages: list[dict] (num_messages,) -- mutated in-place
    tools: list[dict] (num_tools,)    -- OpenAI tool schemas
    tool_handlers: dict[str, Callable]     -- name -> handler
    system: str                        -- system prompt
    max_iterations: int                        -- safety cap
    on_tool_call: Optional[Callable]         -- hook before execution
    """

    # messages: list[dict] (N,) -> list[dict] (N + 2*num_tool_turns,)
    full_messages = (
        [{'role': 'system', 'content': system}] + messages if system else list(messages)
    )

    for _ in range(max_iterations):
        # LLM call
        response = client.chat.completions.create(
            model=MODEL,
            messages=full_messages,
            tools=tools if tools else None,
            tool_choice='auto' if tools else None,
        )

        msg = response.choices[0].message
        reason = response.choices[0].finish_reason

        # Serialise assistant turn for history
        # msg: ChatCompletionMessage -> dict
        serialised = {'role': 'assistant', 'content': msg.content}
        if msg.tool_calls:
            serialised['tool_calls'] = [
                {
                    'id': tc.id,
                    'type': 'function',
                    'function': {'name': tc.function.name,
                                 'arguments': tc.function.arguments},
                }
                for tc in msg.tool_calls
            ]
        full_messages.append(serialised)
        messages.append(serialised)

        if reason != 'tool_calls':
            return messages

        # Tool execution
        for tc in msg.tool_calls:
            name = tc.function.name
            inp = json.loads(tc.function.arguments)

            if on_tool_call:
                on_tool_call(name, inp)

            handler = tool_handlers.get(name)
            if handler:
                try:
                    result = handler(**inp)
                except Exception as exc:
                    result = f'[ToolError] {type(exc).__name__}: {exc}'
            else:
                result = f'[ToolError] No handler for: {name}'

            # Tool results use role='tool' in OpenAI protocol
            tool_msg = {'role': 'tool', 'tool_call_id': tc.id, 'content': str(result)}
            full_messages.append(tool_msg)
            messages.append(tool_msg)

    print(f'[Warning] Reached max_iterations={max_iterations}')
    return messages

In [ ]:
# === Demo ===
def _bash(command, timeout=15):
    try:
        p = subprocess.run(command, shell=True, capture_output=True,
                           text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip() or '(no output)'
    except subprocess.TimeoutExpired:
        return f'[Timeout] exceeded {timeout}s'

DEMO_TOOLS = [{
    'type': 'function',
    'function': {
        'name': 'bash',
        'description': 'Execute a shell command. Returns stdout + stderr.',
        'parameters': {
            'type': 'object',
            'properties': {
                'command': {'type': 'string'},
                'timeout': {'type': 'integer'},
            },
            'required': ['command'],
        },
    },
}]

demo_msgs = [
    {'role': 'user',
     'content': 'What Python version is installed? Use bash.'}
]

run_agent_loop(
    messages=demo_msgs,
    tools=DEMO_TOOLS,
    tool_handlers={'bash': _bash},
    system='You are a concise assistant. Always use tools.',
    on_tool_call=lambda n, i: print(f'  TOOL {n}: {i}'),
)

final = next(
    (m['content'] for m in reversed(demo_msgs)
     if m['role'] == 'assistant' and m['content']),
    '(no reply)',
)
print(f'\nAgent: {final}')

  TOOL bash: {'command': "python --version 2>&1; python3 --version 2>&1; which python || true; which python3 || true; python -c 'import sys; print(sys.version)' 2>&1 || true"}

Agent: Python 3.12.13 is installed (python and python3 point to Python 3.12.13).


# 2) Tool System

Tools are the agent's **sensorimotor interface** to the world. Each tool is two things:
1. A **JSON schema** sent to the LLM -- tells the model what it can call
2. A **handler function** executed locally -- does the actual work

The LLM never executes code directly. It emits a structured request;
the harness executes it. This separation makes permission gating possible.

| Tool | Purpose | Analogy |
|------|---------|---------|
| `bash` | Execute shell commands | Hands |
| `file_read` | Read file contents | Eyes |
| `file_write` | Write / create files | Pen |
| `glob` | Find files by pattern | Map |
| `grep` | Search file contents | Scanner |

```python
glob(pattern='**/*.py')  ->  'src/main.py\nsrc/utils.py'
grep(pattern='def train', include='*.py')  ->  'src/model.py:42:def train(...)'
file_read(file_path='src/main.py', start_line=1, end_line=5)  ->  'import os...'
```

## 2.1 Tool Handlers

In [ ]:
# === 5 core tool handlers ===

def bash_tool(command, timeout=30):
    # command: str, timeout: int -> str (stdout+stderr, max 10_000 chars)
    try:
        p = subprocess.run(command, shell=True, capture_output=True,
                           text=True, timeout=timeout)
        out = p.stdout
        if p.stderr:
            out += '\n[stderr]\n' + p.stderr
        if len(out) > 10_000:
            out = out[:5_000] + '\n...[truncated]...\n' + out[-5_000:]
        return out.strip() or '(no output)'
    except subprocess.TimeoutExpired:
        return f'[Timeout] exceeded {timeout}s'
    except Exception as exc:
        return f'[Error] {exc}'


def file_read_tool(file_path, start_line=None, end_line=None):
    # file_path: str, start_line: int|None, end_line: int|None -> str (max 50_000 chars)
    path = Path(file_path).resolve()
    if not path.exists():
        return f'[Error] Not found: {file_path}'
    if not path.is_file():
        return f'[Error] Not a file: {file_path}'
    text = path.read_text(encoding='utf-8', errors='replace')
    if start_line is not None or end_line is not None:
        lines = text.splitlines(keepends=True)
        text = ''.join(lines[(start_line or 1) - 1 : end_line])
    if len(text) > 50_000:
        text = text[:50_000] + '\n...[truncated at 50KB]'
    return text


def file_write_tool(file_path, content):
    # file_path: str, content: str -> str (confirmation)
    path = Path(file_path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
    return f'Wrote {len(content.encode())} bytes to {file_path}'


def glob_tool(pattern, path='.'):
    # pattern: str, path: str -> str (max 100 paths)
    matches = sorted(glob_module.glob(os.path.join(path, pattern), recursive=True))
    if not matches:
        return 'No files matched.'
    result = '\n'.join(matches[:100])
    if len(matches) > 100:
        result += f'\n...[{len(matches)-100} more omitted]'
    return result


def grep_tool(pattern, path='.', include=None):
    # pattern: str, path: str, include: str|None -> str (max 50 lines)
    cmd = ['grep', '-rn', pattern, path]
    if include:
        cmd += ['--include', include]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        out = r.stdout.strip()
        if not out:
            return f'No matches for: {pattern!r}'
        lines = out.splitlines()
        if len(lines) > 50:
            out = '\n'.join(lines[:50]) + f'\n...[{len(lines)-50} more]'
        return out
    except subprocess.TimeoutExpired:
        return '[Timeout] grep exceeded 10s'
    except FileNotFoundError:
        return '[Error] grep not available'


# === OpenAI tool schemas ===
CORE_TOOL_SCHEMAS = [
    {'type':'function','function':{'name':'bash',
     'description':'Execute a shell command. Use for code, git, packages, navigation.',
     'parameters':{'type':'object',
                   'properties':{'command':{'type':'string'},
                                 'timeout':{'type':'integer','default':30}},
                   'required':['command']}}},
    {'type':'function','function':{'name':'file_read',
     'description':'Read a file. Supports optional line range (1-indexed, inclusive).',
     'parameters':{'type':'object',
                   'properties':{'file_path':{'type':'string'},
                                 'start_line':{'type':'integer'},
                                 'end_line':{'type':'integer'}},
                   'required':['file_path']}}},
    {'type':'function','function':{'name':'file_write',
     'description':'Write content to a file. Creates parent dirs if needed.',
     'parameters':{'type':'object',
                   'properties':{'file_path':{'type':'string'},
                                 'content':{'type':'string'}},
                   'required':['file_path','content']}}},
    {'type':'function','function':{'name':'glob',
     'description':'Find files matching a glob pattern (supports **).',
     'parameters':{'type':'object',
                   'properties':{'pattern':{'type':'string'},
                                 'path':{'type':'string','default':'.'}},
                   'required':['pattern']}}},
    {'type':'function','function':{'name':'grep',
     'description':'Search file contents for a regex pattern with file:line context.',
     'parameters':{'type':'object',
                   'properties':{'pattern':{'type':'string'},
                                 'path':{'type':'string','default':'.'},
                                 'include':{'type':'string'}},
                   'required':['pattern']}}},
]

CORE_TOOL_HANDLERS = {
    'bash': bash_tool,
    'file_read': file_read_tool,
    'file_write': file_write_tool,
    'glob': glob_tool,
    'grep': grep_tool,
}

names = [t['function']['name'] for t in CORE_TOOL_SCHEMAS]
print(f'Registered {len(names)} core tools: {names}')

Registered 5 core tools: ['bash', 'file_read', 'file_write', 'glob', 'grep']


In [ ]:
# === Demo ===
demo_msgs = [
    {'role':'user',
     'content':'Use glob to list Python files here, then read lines 1-3 of one.'}
]

run_agent_loop(
    messages=demo_msgs, tools=CORE_TOOL_SCHEMAS, tool_handlers=CORE_TOOL_HANDLERS,
    system='You are a concise file-exploration assistant.',
    on_tool_call=lambda n,i: print(f'  TOOL {n}: {str(i)[:80]}'),
)

final = next(
    (m['content'] for m in reversed(demo_msgs)
    if m['role']=='assistant' and m['content']), '(no reply)'
)

print(f'\nAgent:\n{final[:300]}')

  TOOL glob: {'pattern': '**/*.py', 'path': '.'}

Agent:
I ran glob "**/*.py" in the current directory — no Python files were found.

Options:
- Tell me a different path or glob pattern to try.
- I can search for executable scripts with a Python shebang (#!/usr/bin/env python or #!/usr/bin/python) — want me to run that search?
- Or I can list all files so


## 2.2 TodoWrite and Planning

Without explicit planning, LLMs start executing immediately and lose track
of the overall goal as context fills up with tool outputs.
**TodoWrite** forces the model to decompose tasks into a checklist *before* acting.

A **nag reminder** is injected every N tool calls:
`'You have 3 pending tasks -- update their status.'`

This compensates for the model forgetting its plan once buried in history.
Taken directly from Claude Code's `TodoWriteTool` and `queryProcessor.ts`.

```python
todo(action='write', tasks=[{description:'Read tests'},{description:'Write fix'}])
# -> 'Created 2 tasks. IDs: todo_1, todo_2'

todo(action='update', todo_id='todo_1', status='done')
# -> 'Updated todo_1 -> done'

todo(action='read')
# -> 'OK [todo_1] Read tests (done)'
# -> '-- [todo_2] Write fix (pending)'
```

- `write()` clears and recreates the plan -- plan-from-scratch semantics.
- `update()` transitions a task through the status FSM.
- `get_nag_reminder()` returns a reminder string every N calls; None otherwise.

In [ ]:
@dataclass
class TodoItem:
    id: str
    description: str
    status: str = 'pending'  # pending | in_progress | done | blocked


class TodoManager:
    # Manages an in-memory structured todo list for agent planning.
    # State is kept in the harness (not in conversation history) to avoid
    # polluting the context with raw CRUD records.

    _ICON = {'pending':'--', 'in_progress':'>>', 'done':'OK', 'blocked':'XX'}

    def __init__(self):
        # _items: dict[str, TodoItem] (num_todos,)
        self._items = {}
        self._counter = 0

    def write(self, tasks):
        # tasks: list[dict] (num_tasks,) -> str (confirmation)
        self._items.clear()
        self._counter = 0
        for task in tasks:
            self._counter += 1
            tid = f'todo_{self._counter}'
            self._items[tid] = TodoItem(id=tid, description=task.get('description',''))
        return f'Created {len(tasks)} task(s). IDs: {", ".join(self._items)}'

    def update(self, todo_id, status):
        # todo_id: str, status: str -> str (confirmation | error)
        valid = {'pending','in_progress','done','blocked'}
        if todo_id not in self._items:
            return f'[Error] Unknown id: {todo_id}'
        if status not in valid:
            return f'[Error] Invalid status. Valid: {valid}'
        self._items[todo_id].status = status
        return f'Updated {todo_id} -> {status}'

    def read(self):
        # -> str (formatted list)
        if not self._items:
            return 'No tasks.'
        return '\n'.join(
            f'{self._ICON.get(i.status,"?")}'
            f' [{i.id}] {i.description} ({i.status})'
            for i in self._items.values()
        )

    def get_nag_reminder(self, call_count, interval=5):
        # call_count: int, interval: int -> Optional[str]
        if call_count == 0 or call_count % interval != 0:
            return None
        pending = [i for i in self._items.values()
                   if i.status in {'pending','in_progress'}]
        if not pending:
            return None
        return (
            f'[Reminder] {len(pending)} task(s) unfinished. '
            f'Update status as you complete them.\n{self.read()}'
        )


def build_todo_handler(manager):
    # Factory: returns a dispatched handler bound to manager.
    # manager: TodoManager -> Callable (action, tasks?, todo_id?, status? -> str)
    def handler(action, tasks=None, todo_id=None, status=None):
        if action == 'write':  return manager.write(tasks or [])
        if action == 'update': return manager.update(todo_id or '', status or '')
        if action == 'read':   return manager.read()
        return f'[Error] Unknown action: {action}'
    return handler


TODO_TOOL_SCHEMA = {
    'type':'function','function':{
        'name':'todo',
        'description':
            'Manage a structured task list. ALWAYS call todo(action=write) at the start '
            'of any complex task to create a plan. Call update as you complete each step.',
        'parameters':{'type':'object',
            'properties':{
                'action':{'type':'string','enum':['write','update','read']},
                'tasks':{'type':'array','items':{'type':'object',
                         'properties':{'description':{'type':'string'}}}},
                'todo_id':{'type':'string'},
                'status':{'type':'string','enum':['pending','in_progress','done','blocked']},
            },
            'required':['action'],
        },
    },
}

# === Demo ===
_mgr = TodoManager()
print(_mgr.write([{'description':'Read codebase'},
                  {'description':'Find the bug'},
                  {'description':'Write the fix'},
                  {'description':'Run tests'}]))
_mgr.update('todo_1', 'done')
_mgr.update('todo_2', 'in_progress')
print(_mgr.read())
print('\nNag at call #5:', _mgr.get_nag_reminder(5))

Created 4 task(s). IDs: todo_1, todo_2, todo_3, todo_4
OK [todo_1] Read codebase (done)
>> [todo_2] Find the bug (in_progress)
-- [todo_3] Write the fix (pending)
-- [todo_4] Run tests (pending)

Nag at call #5: [Reminder] 3 task(s) unfinished. Update status as you complete them.
OK [todo_1] Read codebase (done)
>> [todo_2] Find the bug (in_progress)
-- [todo_3] Write the fix (pending)
-- [todo_4] Run tests (pending)


## 2.3 Subagent Spawning

As a task grows, the main agent context fills with file reads, bash outputs,
and intermediate reasoning -- all noise from the perspective of a focused subtask.
Performance degrades: the model attends to stale data and makes contradictory decisions.

**Subagents solve this by context isolation.** When the main agent encounters
a well-scoped subtask, it delegates via the `agent` tool. The child receives:

- A **fresh `messages[]`** -- zero inherited noise from the parent
- A focused task description as its first user message
- The same tool set (or a restricted subset)

The child runs to completion; its final reply is returned to the parent
as a tool result. The child conversation is then discarded.

```python
# Parent calls:
agent(task='Read src/auth.py and explain validate_token.')
# -> 'validate_token decodes the JWT, checks exp, verifies signature...'
```

Implementation is a **recursive call** to `run_agent_loop` with a new `messages[]`.
The factory pattern lets callers configure which tools the child receives.

In [ ]:
def build_agent_tool(parent_system, tools, tool_handlers, max_iterations=10):
    """
    Factory: creates an agent tool handler bound to a specific tool set.

    parent_system:      str -- inherited context for domain continuity
    tools:              list[dict] (num_tools,) -- schemas for child
    tool_handlers:      dict[str, Callable]    -- handlers for child
    -> Callable:        task: str -> str (child final text)
    """

    def agent_handler(task):
        '''
        task: str -> str (subagent final reply | error)
        Fresh messages[] -- the core isolation mechanism
        '''

        child_msgs = [{'role': 'user', 'content': task}]
        child_system = (
            f'{parent_system}\n\n'
            'You are a focused subagent. Complete the task thoroughly and '
            'return a clear self-contained summary. Do not ask follow-up questions.'
        )
        try:
            run_agent_loop(
                messages=child_msgs, tools=tools, tool_handlers=tool_handlers,
                system=child_system, max_iterations=max_iterations,
            )
        except Exception as exc:
            return f'[SubagentError] {type(exc).__name__}: {exc}'

        # child_msgs: list[dict] (N,) -> str (last assistant text)
        return next(
            (m['content'] for m in reversed(child_msgs)
             if m['role'] == 'assistant' and m['content']),
            '(subagent produced no output)',
        )

    return agent_handler


AGENT_TOOL_SCHEMA = {
    'type':'function','function':{
        'name':'agent',
        'description':
            'Spawn a focused subagent with a fresh context to handle one subtask. '
            'Use when a subtask is well-scoped and benefits from isolated execution.',
        'parameters':{'type':'object',
            'properties':{'task':{'type':'string',
                'description':'A clear self-contained description of the subtask.'}},
            'required':['task'],
        },
    },
}


## 2.4 Skill Loading

A skill is a **reusable knowledge file** (SKILL.md) covering a specific domain:
how to run the test suite, the git branching convention, the API style guide.
Rather than injecting all domain knowledge at startup (wasting tokens on
knowledge the agent may never need), skills are loaded **on demand** via a tool.

The model calls `list_skills` to discover available skills, then
`load_skill(name='pytest')` to inject one into context.
Equivalent to RAG over a curated knowledge base, but simpler and more controllable.


```python
list_skills()  ->  'Available skills: docker, git, pytest, sqlalchemy'

load_skill(name='pytest') ->  '# pytest Skill\n## How to run\npytest -xvs tests/\n...'
```

```
# Skill Name
## Context -- when to use this skill
## Instructions -- step-by-step guidance
## Examples -- concrete illustrations
```

In [ ]:
class SkillLibrary:
    '''
    Manages a directory of SKILL.md knowledge files.
    Skills are loaded on demand to minimise baseline context token usage.
    '''

    def __init__(self, skills_dir='skills'):
        self.dir = Path(skills_dir)
        self.dir.mkdir(exist_ok=True)

        # Creating demo skill
        self._seed()

    def _seed(self):
        # Write demo skills if directory is empty
        demos = {
            'git.md': (
                '# git Skill\n'
                '## Context\nUse for git: commits, branches, diffs, history.\n'
                '## Key Commands\n'
                '- git log --oneline -10  -- recent commits\n'
                '- git diff HEAD~1        -- last commit diff\n'
                '- git status             -- working tree state\n'
                '## Conventions\nCheck git status before changes. '
                'Use imperative commit messages.\n'
            ),
            'pytest.md': (
                '# pytest Skill\n'
                '## Context\nUse when running or writing Python tests.\n'
                '## Running\n'
                '- pytest -xvs          -- stop on first failure, verbose\n'
                '- pytest tests/foo.py::test_bar  -- single test\n'
                '## Writing\nUse fixtures for shared setup. '
                'Parametrize for data-driven tests.\n'
            ),
            'docker.md': (
                '# docker Skill\n'
                '## Context\nUse for Docker container operations.\n'
                '## Key Commands\n'
                '- docker build -t name:tag .         -- build image\n'
                '- docker run --rm -it name:tag bash  -- interactive shell\n'
                '- docker logs container_id           -- view logs\n'
            ),
        }
        for fname, content in demos.items():
            p = self.dir / fname
            if not p.exists():
                p.write_text(content, encoding='utf-8')

    def list_skills(self):
        # str (comma-separated skill names | 'No skills available.')
        paths = sorted(self.dir.glob('*.md'))
        if not paths:
            return 'No skills available.'
        return 'Available skills: ' + ', '.join(p.stem for p in paths)

    def load_skill(self, name):
        # name: str -> str (content | error with available list)
        stem = name.removesuffix('.md')
        p = self.dir / f'{stem}.md'
        if not p.exists():
            return f'[Error] Skill "{name}" not found. {self.list_skills()}'
        return p.read_text(encoding='utf-8')


def build_skill_tools(lib):
    # lib: SkillLibrary -> (schemas: list[dict] (2,), handlers: dict[str, Callable] (2,))
    schemas = [
        {'type':'function','function':{
            'name':'list_skills',
            'description':'List all available domain knowledge skills.',
            'parameters':{'type':'object','properties':{}},
        }},
        {'type':'function','function':{
            'name':'load_skill',
            'description':
                'Load a skill into context. Call list_skills first to see what is available.',
            'parameters':{'type':'object',
                'properties':{'name':{'type':'string',
                    'description':"Skill name e.g. 'pytest', 'git', 'docker'."}},
                'required':['name'],
            },
        }},
    ]
    handlers = {'list_skills': lib.list_skills, 'load_skill': lib.load_skill}
    return schemas, handlers

# -- Demo -------------------------------------------------------------------
_lib = SkillLibrary()
print(_lib.list_skills())

print('\n--- pytest skill preview ---')
print(_lib.load_skill('pytest')[:220])

Available skills: docker, git, pytest

--- pytest skill preview ---
# pytest Skill
## Context
Use when running or writing Python tests.
## Running
- pytest -xvs          -- stop on first failure, verbose
- pytest tests/foo.py::test_bar  -- single test
## Writing
Use fixtures for shared s


## 2.6 Autonomous Skill Creation

THe previous skills are **loaded** on demand. Hermes adds the inverse:
after completing a complex task the agent **writes a new skill** capturing
what it just learned. The skill library grows from experience.

This is the closed learning loop:

```
Task completed
     |
     v
reflect_on_task()   <- asks LLM: what reusable procedure did I just follow?
     |
     v
write_skill()       <- saves SKILL.md to skills/ directory
     |
     v
Next time a similar task arrives, load_skill() finds and injects it
```

**Key design decision:** the reflection call is cheap (one short LLM call)
and gated on task complexity -- trivial tasks do not generate skills.
Only tasks that involved 3+ tool calls and produced a non-trivial result
are worth encoding.

**Sample Output:**
```
Task: 'Find all TODO comments in the codebase and summarise them'
Generated skill: skills/auto_todo_audit.md
  # TODO Audit Skill
  ## When to use
  When asked to audit or summarise TODO/FIXME/HACK comments.
  ## Steps
  1. grep -rn 'TODO\|FIXME\|HACK' --include='*.py'
  2. Group by file, count per category
  3. Summarise: N todos, M fixmes, top files are ...
```

In [ ]:
class AutonomousSkillCreator:
    '''
    After a complex task, asks the LLM to reflect and write a new SKILL.md.
    Extends SkillLibrary -- skills are loaded and written by the same system.
    Modelled on Hermes agent skill auto-generation loop.
    '''

    MIN_TOOL_CALLS = 3
    MAX_SKILL_TOKENS = 300

    def __init__(self, skill_lib):
        # skill_lib: SkillLibrary
        self._lib      = skill_lib

        # _generated: list[str] (skill names created this session)
        self._generated: list = []

    def should_reflect(self, messages, tool_call_count):
        '''
        Decide whether this task warrants skill creation.
        Heuristic: enough tool calls + non-trivial final reply.
        messages: list[dict] (N,), tool_call_count: int -> bool
        '''

        if tool_call_count < self.MIN_TOOL_CALLS:
            return False
        last_reply = next(
            (m['content'] for m in reversed(messages)
             if m.get('role') == 'assistant' and m.get('content')),
            '',
        )

        # Skip if the agent just said one sentence
        return len(last_reply.split()) > 20

    def reflect_and_write(self, task_description, messages, tool_call_count):
        '''
        Full reflection pipeline: assess -> generate -> save.
        task_description: str, messages: list[dict] (N,) -> Optional[str] (skill name)
        '''
        if not self.should_reflect(messages, tool_call_count):
            return None

        # Summarise what the agent did into a reusable procedure
        # task + messages: context -> skill_content: str (SKILL.md text)
        history_snippet = json.dumps(messages[-6:], default=str)[:3_000]
        resp = client.chat.completions.create(
            model=MODEL,
            max_completion_tokens=self.MAX_SKILL_TOKENS,
            messages=[
                {
                    'role': 'system',
                    'content': (
                        'You extract reusable skills from agent task transcripts. '
                        'Write a concise SKILL.md with sections: '
                        '# Skill Name, ## When to use, ## Steps (numbered), ## Notes. '
                        'Focus on the repeatable procedure, not this specific task. '
                        'Be concrete. Under 200 words.'
                    ),
                },
                {
                    'role': 'user',
                    'content': f'Task: {task_description}\n\nTranscript:\n{history_snippet}',
                },
            ],
        )
        skill_content = resp.choices[0].message.content

        if not skill_content or not skill_content.strip():
            print(f'  [SkillCreator] LLM returned empty skill content for task: {task_description}')
            return None

        # Extract skill name from the first heading line
        # skill_content: str -> skill_name: str
        first_line = skill_content.strip().splitlines()[0]
        raw_name   = first_line.replace('#', '').strip().lower()
        skill_name = 'auto_' + raw_name.replace(' ', '_')[:30]

        # Write to the skill library -- immediately available for future tasks
        (self._lib.dir / f'{skill_name}.md').write_text(skill_content, encoding='utf-8')
        self._generated.append(skill_name)
        print(f'  [SkillCreator] Generated skill: {skill_name}.md')
        return skill_name

    @property
    def generated_this_session(self):
        return list(self._generated)


# -- Demo -------------------------------------------------------------------
_lib2     = SkillLibrary()   # reuse from §5
_creator  = AutonomousSkillCreator(_lib2)

# Simulate a completed multi-tool task
fake_messages = [
    {'role': 'user',      'content': 'Find all TODO comments in the repo and summarise them.'},
    {'role': 'assistant', 'content': None,
     'tool_calls': [{'id':'tc1','type':'function','function':{'name':'bash','arguments':'{"command":"grep -rn TODO ."}'}}]},
    {'role': 'tool',      'tool_call_id': 'tc1', 'content': 'src/main.py:12: # TODO fix auth\nsrc/utils.py:44: # TODO add logging'},
    {'role': 'assistant', 'content': None,
     'tool_calls': [{'id':'tc2','type':'function','function':{'name':'bash','arguments':'{"command":"grep -rn FIXME ."}'}}]},
    {'role': 'tool',      'tool_call_id': 'tc2', 'content': 'src/db.py:88: # FIXME connection leak'},
    {'role': 'assistant', 'content': 'Found 2 TODOs and 1 FIXME. TODOs: fix auth (main.py:12), add logging (utils.py:44). FIXME: connection leak (db.py:88). Recommend addressing the connection leak first as it is a bug.'},
]

skill_name = _creator.reflect_and_write(
    task_description='Find all TODO comments in the repo and summarise them.',
    messages=fake_messages,
    tool_call_count=3,
)

print(f'Generated: {skill_name}')
print(f'Skills now available: {_lib2.list_skills()}')

  [SkillCreator] LLM returned empty skill content for task: Find all TODO comments in the repo and summarise them.
Generated: None
Skills now available: Available skills: docker, git, pytest


## 2.6 Context Compaction

Every tool call grows `messages[]`. A `file_read` on a 500-line file costs
~2 000 tokens. After reading 20 files and running 15 bash commands,
you have consumed 50 000+ tokens. Without compaction the session overflows.

1. TRUNCATE (cheap, lossy)
  - Replace old tool result content with '[truncated]'.
  - Preserves structure (model knows the call happened) but drops the data.

2. SUMMARISE (one extra API call, near-lossless)
  - Ask the model to compress old turns into 2-3 paragraphs.
  - Semantic content survives; verbatim history is discarded.

3. HARD RESET (cheap, last resort)
  - Keep only the last N messages. Guarantees fit within any budget.

```python
# 80 000-token conversation
compact_messages(messages, token_budget=40_000)
# -> [summary_pair (2 msgs), ...last 6 msgs]  ~8 000 tokens
```

In [ ]:
def estimate_tokens(messages):
    '''
    Exact token count using tiktoken
    messages: list[dict] (num_messages,) -> int
    '''

    encoding = tiktoken.get_encoding("cl100k_base")
    num_tokens = 0
    for m in messages:
        c = m.get('content') or ''
        text = c if isinstance(c, str) else json.dumps(c)
        num_tokens += len(encoding.encode(text))
        if 'tool_calls' in m:
            num_tokens += len(encoding.encode(json.dumps(m['tool_calls'])))
    return num_tokens

def _truncate_old_tool_results(messages, preserve_last_n):
    '''
    Layer 1: replace content in old tool-result messages with a placeholder.
    messages: list[dict] (N,) -> list[dict] (N,) -- same length, less content
    '''
    cutoff = max(0, len(messages) - preserve_last_n)
    result = []
    for i, m in enumerate(messages):
        if i < cutoff and m.get('role') == 'tool':
            result.append({**m, 'content': '[output truncated during compaction]'})
        else:
            result.append(m)
    return result


def _summarise_old_history(messages, preserve_last_n):
    '''
    Layer 2: compress old turns into a 2-message summary pair.
    One extra LLM call; semantic content survives.
    messages: list[dict] (N,) -> list[dict] (2 + preserve_last_n,)
    '''

    cutoff  = max(0, len(messages) - preserve_last_n)
    old     = messages[:cutoff]
    recent  = messages[cutoff:]
    if not old:
        return messages
    history_text = json.dumps(old, indent=2, default=str)[:8_000]
    resp = client.chat.completions.create(
        model=MODEL,
        max_completion_tokens=400,
        messages=[
            {'role':'system','content':'Summarise this agent conversation in 2-3 paragraphs. Cover: goals accomplished, findings, decisions made, pending items.'},
            {'role':'user','content': history_text},
        ],
    )
    summary = resp.choices[0].message.content
    return [
        {'role':'user',      'content': f'[Session summary]\n{summary}'},
        {'role':'assistant', 'content': 'Understood. Continuing from this context.'},
    ] + recent


def compact_messages(
    messages,
    system='',
    token_budget=60_000,
    trigger_ratio=0.8,
    context_window=128_000,
    preserve_last_n=6,
):
    '''
    3-layer context compaction. Triggered when token usage exceeds
    trigger_ratio * context_window. Layers applied in order, stopping when budget met.
    messages: list[dict] (N,) -> list[dict] (N_prime,) where N_prime <= N
    '''
    threshold = int(context_window * trigger_ratio)
    current   = estimate_tokens(messages)
    if current <= threshold:
        return messages
    print(f'  [Compact] {current:,} tokens > threshold {threshold:,}. Compacting...')

    # Layer 1: truncate old tool outputs
    compacted = _truncate_old_tool_results(messages, preserve_last_n)
    after_l1  = estimate_tokens(compacted)
    print(f'  [Compact] After layer 1 (truncate): {after_l1:,} tokens')

    # Layer 2: summarise if still over budget
    if after_l1 > token_budget:
        compacted = _summarise_old_history(compacted, preserve_last_n)
        after_l2  = estimate_tokens(compacted)
        print(f'  [Compact] After layer 2 (summarise): {after_l2:,} tokens')

    # Layer 3: hard reset -- keep only the most recent messages
    if estimate_tokens(compacted) > token_budget:
        compacted = compacted[-preserve_last_n:]
        print(f'  [Compact] After layer 3 (hard reset): {estimate_tokens(compacted):,} tokens')

    return compacted

# -- Demo -------------------------------------------------------------------
bloated = [{'role':'user', 'content':'Analyse the codebase.'}]
for i in range(8):
    bloated.append({'role':'tool','tool_call_id':f'tc_{i}','content':'X'*2_000})
print(f'Before: ~{estimate_tokens(bloated):,} tokens')
compacted = compact_messages(bloated, token_budget=2_000, trigger_ratio=0.001)
print(f'After:  ~{estimate_tokens(compacted):,} tokens')


Before: ~2,006 tokens
  [Compact] 2,006 tokens > threshold 128. Compacting...
  [Compact] After layer 1 (truncate): 1,520 tokens
After:  ~1,520 tokens


# 3) Task System

`TodoManager` is ephemeral -- it dies with the process.
For long-running work spanning hours or multiple sessions, tasks must be **persisted to disk** and ordered by **dependencies**.

The Task System adds:
1. **Persistence** -- tasks stored as JSONL (append-friendly, crash-safe)
2. **Dependency graph** -- task B cannot start until task A is done
3. **Status FSM** -- `pending -> in_progress -> done | failed`
4. **Topological ordering** -- Kahns algorithm for valid execution sequences

```python
t1 = store.create(TaskSpec('Write tests'))
t2 = store.create(TaskSpec('Fix bug', deps=['task_1']))
store.get_ready() # -> [task_1](task_2 blocked on task_1)
store.transition('task_1', 'done')
store.get_ready() # -> [task_2](now unblocked)
```

JSONL is preferred over SQLite: zero extra dependencies, human-readable,
and a partial write corrupts only the last line -- all previous tasks survive.

## 3.1 Task Store

In [ ]:
@dataclass
class TaskSpec:
    description: str
    deps: list = field(default_factory=list)
    metadata: dict = field(default_factory=dict)

@dataclass
class Task:
    id: str
    description: str
    status: str   # pending | in_progress | done | failed
    deps: list
    created_at: str
    updated_at: str
    metadata: dict = field(default_factory=dict)

    # Valid FSM transitions
    _VALID: dict = field(
        default_factory=lambda: {
            'pending':     {'in_progress'},
            'in_progress': {'done','failed','pending'},
            'done':        set(),
            'failed':      {'pending'},
            },

        repr=False
    )

    def can_transition(self, new): return new in self._VALID.get(self.status, set())
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

class FileTaskStore:
    '''
    File-backed task store. Persists to JSONL; loads into in-memory index.
    Write-through: every mutation immediately flushes to disk.
    '''

    def __init__(self, store_path='.tasks/tasks.jsonl'):
        self.path = Path(store_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        # _tasks: dict[str, Task] (num_tasks,)
        self._tasks = {}
        self._load()

    def _load(self):
        if not self.path.exists(): return
        for line in self.path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                d = json.loads(line)
                self._tasks[d['id']] = Task(**{k:v for k,v in d.items() if not k.startswith('_')})

    def _flush(self):
        lines = [json.dumps(t.to_dict()) for t in self._tasks.values()]
        self.path.write_text(('\n'.join(lines)+'\n') if lines else '', encoding='utf-8')

    def create(self, spec):
        # spec: TaskSpec -> Task (status='pending')
        tid = f'task_{len(self._tasks)+1}'
        now = datetime.now().isoformat()
        task = Task(
            id=tid,
            description=spec.description,
            status='pending',
            deps=spec.deps,
            created_at=now,
            updated_at=now,
            metadata=spec.metadata
        )
        self._tasks[tid] = task
        self._flush()
        return task

    def get(self, task_id): return self._tasks.get(task_id)

    def transition(self, task_id, new_status):
        # task_id: str, new_status: str -> str (confirmation | error)
        t = self._tasks.get(task_id)
        if not t: return f'[Error] Unknown task: {task_id}'
        if not t.can_transition(new_status):
            return f'[Error] {t.status} -> {new_status} is not a valid transition.'
        t.status = new_status
        t.updated_at = datetime.now().isoformat()
        self._flush()
        return f'{task_id} -> {new_status}'

    def get_ready(self):
        # -> list[Task] (num_ready,): pending tasks with all deps done
        done_ids = {t.id for t in self._tasks.values() if t.status == 'done'}
        return [
            t for t in self._tasks.values()
            if t.status == 'pending' and all(d in done_ids for d in t.deps)
        ]

    def topological_order(self):

        # Kahn's algorithm -- valid execution order respecting dependencies.
        # list[str] (num_tasks,)  |  raises ValueError on circular deps

        in_deg = {tid: 0 for tid in self._tasks}
        for t in self._tasks.values():
            for dep in t.deps:
                if dep in in_deg: in_deg[t.id] += 1
        queue = deque(tid for tid, d in in_deg.items() if d == 0)
        order = []
        while queue:
            tid = queue.popleft()
            order.append(tid)
            for t in self._tasks.values():
                if tid in t.deps:
                    in_deg[t.id] -= 1
                    if in_deg[t.id] == 0: queue.append(t.id)
        if len(order) != len(self._tasks): raise ValueError('Circular dependency detected.')
        return order

    def summary(self):
        icons = {'pending':'--','in_progress':'>>','done':'OK','failed':'XX'}
        return '\n'.join(
            f'{icons.get(t.status,"?")} [{t.id}] {t.description} | deps: {t.deps or "none"}'
            for t in self._tasks.values()) or 'No tasks.'


# -- Demo -------------------------------------------------------------------
store = FileTaskStore('.tasks/demo.jsonl')
store._tasks.clear(); store._flush()
store.create(TaskSpec('Read codebase'))
store.create(TaskSpec('Write tests',  deps=['task_1']))
store.create(TaskSpec('Fix bug',      deps=['task_1']))
store.create(TaskSpec('Run CI',       deps=['task_2','task_3']))

print(store.summary())
print('\nReady now: ', [t.id for t in store.get_ready()])
print('Topological order:', store.topological_order())
print(store.transition('task_1','done'))
print('Ready after t1:  ', [t.id for t in store.get_ready()])

-- [task_1] Read codebase | deps: none
-- [task_2] Write tests | deps: ['task_1']
-- [task_3] Fix bug | deps: ['task_1']
-- [task_4] Run CI | deps: ['task_2', 'task_3']

Ready now:  ['task_1']
Topological order: ['task_1', 'task_2', 'task_3', 'task_4']
[Error] pending -> done is not a valid transition.
Ready after t1:   ['task_1']


## 3.2 Background Tasks

Some operations take a long time: running a full test suite, compiling a
large project, fetching a remote resource. If the agent **blocks** on these,
it wastes context tokens and the user waits idly.

Background tasks execute slow commands in daemon threads while the agent
loop continues. On completion, results are pushed to a `NotificationQueue`.
Before each LLM call, the harness drains the queue and injects pending
notifications as user messages.

```python
runner.run('pytest tests/ -q', task_id='test_run')
# -> '[Background] test_run started.'
# ... 30 seconds later, injected into conversation:
# '[Notification] test_run completed in 28.3s: 47 passed'
```

```
Main thread (agent loop)
  |-- Spawns BackgroundTaskRunner daemon threads
  |     \-- On completion -> push BackgroundResult to NotificationQueue
  \-- drain_notifications() called in _pre_turn()
        \-- Injects pending results as user messages
```

In [ ]:
@dataclass
class BackgroundResult:
    task_id: str
    command: str
    exit_code: int
    output: str
    duration_s: float
    completed_at: str = field(default_factory=lambda: datetime.now().isoformat())


class NotificationQueue:
    # Thread-safe queue for background task results.
    # Lock+list suffices because we only drain from the main thread.

    def __init__(self):
        # _pending: list[BackgroundResult] (num_completed,)
        self._pending = []
        self._lock = threading.Lock()

    def push(self, result):
        # Called from background thread on task completion.
        with self._lock:
            self._pending.append(result)

    def drain(self):
        # Atomically consume all pending notifications.
        # Called from main thread before each LLM call.
        # -> list[BackgroundResult] (num_pending,)
        with self._lock:
            results, self._pending = list(self._pending), []
        return results

    def format_and_drain(self):
        # Drain and format all notifications as a single string.
        # -> str (formatted notifications | '')
        results = self.drain()
        if not results: return ''
        lines = []
        for r in results:
            status = 'completed' if r.exit_code == 0 else f'failed (exit {r.exit_code})'
            preview = r.output[:200].replace('\n', ' ')
            lines.append(f"[Notification] '{r.task_id}' {status} in {r.duration_s:.1f}s: {preview}")
        return '\n'.join(lines)


class BackgroundTaskRunner:
    # Executes shell commands in daemon threads and notifies on completion.
    # Daemon threads are reaped automatically when the main process exits.

    def __init__(self, queue):
        self._queue = queue
        # _threads: dict[str, threading.Thread] (num_active,)
        self._threads = {}

    def run(self, command, task_id=None):
        # command: str, task_id: str|None -> str (immediate acknowledgement)
        task_id = task_id or f'bg_{uuid.uuid4().hex[:6]}'

        def _worker(cmd, tid):
            start = time.time()
            try:
                p = subprocess.run(cmd, shell=True, capture_output=True,
                                   text=True, timeout=300)
                output   = (p.stdout + p.stderr).strip()
                exit_code = p.returncode
            except subprocess.TimeoutExpired:
                output, exit_code = '[Timeout] exceeded 300s', -1
            except Exception as exc:
                output, exit_code = f'[Error] {exc}', -1
            self._queue.push(BackgroundResult(
                task_id=tid, command=cmd, exit_code=exit_code,
                output=output[:2_000], duration_s=round(time.time()-start, 2),
            ))

        t = threading.Thread(target=_worker, args=(command, task_id), daemon=True)
        t.start()
        self._threads[task_id] = t
        return f"[Background] '{task_id}' started. Result arrives via notification."

    def is_running(self, task_id):
        t = self._threads.get(task_id)
        return t is not None and t.is_alive()


# -- Demo -------------------------------------------------------------------
_nq = NotificationQueue()
_runner = BackgroundTaskRunner(_nq)
print(_runner.run("sleep 1 && echo 'tests: 42 passed'", task_id='test_run'))
print('Agent continues working while tests run...')
time.sleep(1.5)
notifs = _nq.format_and_drain()
print(f'\nNotifications received:\n{notifs}' if notifs else 'No notifications yet.')

[Background] 'test_run' started. Result arrives via notification.
Agent continues working while tests run...

Notifications received:
[Notification] 'test_run' completed in 1.0s: tests: 42 passed


# 4) Agent Teams

Some tasks exceed what a single-context agent can handle in scope,
parallelism, or specialisation. Agent teams use a **coordinator + workers** pattern.

The coordinator decomposes the goal and uses the `delegate` tool to assign subtasks.
Workers run as subagents with specialised system prompts. The coordinator synthesises results.

**JSONL Mailboxes**

Workers communicate via **append-only JSONL mailboxes** -- one file per agent.
Append-only is crash-safe: a mid-write process kill leaves all prior messages intact.
Incremental reading via a byte offset avoids re-processing old messages.

```
Coordinator goal: 'Investigate the auth bug AND write regression tests.'
  -> delegate(worker_id='investigator', task='Find root cause of auth bug')
  -> delegate(worker_id='tester',       task='Write regression tests')
  -> Synthesises: 'Root cause: ...  Tests added: ...'
```

## 4.1 Coordinator

In [ ]:
@dataclass
class TeamMessage:
    from_agent: str
    to_agent:   str
    msg_type:   str  # task | result | plan | approve | reject | shutdown
    content:    str
    msg_id:     str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    timestamp:  str = field(default_factory=lambda: datetime.now().isoformat())

class Mailbox:
    # Append-only JSONL mailbox for one agent.
    # Incremental reading via stored byte-offset cursor.

    def __init__(self, agent_id, mailbox_dir='.mailboxes'):
        self.agent_id = agent_id
        self.path = Path(mailbox_dir) / f'{agent_id}.jsonl'
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._offset = 0  # byte offset for incremental reads

    def send(self, msg):
        # msg: TeamMessage -> None (appends to recipient file)
        recipient = self.path.parent / f'{msg.to_agent}.jsonl'
        with open(recipient, 'a', encoding='utf-8') as f:
            f.write(json.dumps(msg.__dict__) + '\n')

    def receive_new(self):
        # -> list[TeamMessage] (num_new,) -- messages since last call
        if not self.path.exists(): return []
        content = self.path.read_text(encoding='utf-8')
        new = content[self._offset:]
        self._offset = len(content)
        msgs = []
        for line in new.splitlines():
            if line.strip(): msgs.append(TeamMessage(**json.loads(line)))
        return msgs

    def clear(self):
        if self.path.exists(): self.path.unlink()
        self._offset = 0


class TeamMember:
    # A worker agent with its own identity, mailbox, and tool set.

    def __init__(self, agent_id, role, tools, tool_handlers, mailbox_dir='.mailboxes'):
        self.agent_id    = agent_id
        self.role        = role
        self.mailbox     = Mailbox(agent_id, mailbox_dir)
        self.tools       = tools
        self.tool_handlers = tool_handlers

    def execute_task(self, task_description):
        # Fresh messages[] per task -- context isolation (section 4)
        # task_description: str -> str (result text)
        messages = [{'role': 'user', 'content': task_description}]
        run_agent_loop(
            messages=messages,
            tools=self.tools,
            tool_handlers=self.tool_handlers,
            system=f"You are agent '{self.agent_id}', a {self.role}. Be thorough and concise.",
        )
        return next(
            (m['content'] for m in reversed(messages)
            if m['role']=='assistant' and m['content']),
            '(no output)'
        )

def run_coordinator(goal, workers, coordinator_tools, coordinator_handlers):
    # Run a coordinator that decomposes goal and delegates to workers.
    # goal: str, workers: list[TeamMember] (num_workers,) -> str (final synthesis)
    worker_map = {w.agent_id: w for w in workers}

    def delegate(worker_id, task):
        # Dispatch a task to a named worker and return their result.
        w = worker_map.get(worker_id)
        if not w: return f'[Error] Unknown worker: {worker_id}. Available: {list(worker_map)}'
        print(f'  [Coord -> {worker_id}] {task[:60]}...')
        result = w.execute_task(task)
        print(f'  [{worker_id} -> Coord] {result[:60]}...')
        return result

    delegate_schema = {
        'type':'function','function':{
            'name':'delegate',
            'description':'Assign a focused task to a worker agent and receive their result.',
            'parameters':{'type':'object',
                'properties':{
                    'worker_id':{'type':'string','enum':[w.agent_id for w in workers]},
                    'task':{'type':'string'},
                },
                'required':['worker_id','task'],
            },
        },
    }

    roster = '\n'.join(f'- {w.agent_id}: {w.role}' for w in workers)
    system = (
        'You are the COORDINATOR. Decompose the goal into subtasks and delegate '
        'each to the appropriate worker. Do NOT do the work yourself. '
        'After all workers report, synthesise a final answer.\n\n'
        f'Available workers:\n{roster}'
    )
    messages = [{'role':'user','content': goal}]
    run_agent_loop(
        messages=messages, tools=coordinator_tools+[delegate_schema],
        tool_handlers={**coordinator_handlers,'delegate':delegate}, system=system,
    )
    return next(
        (m['content'] for m in reversed(messages)
        if m['role']=='assistant' and m['content']),
        '(no output)'
    )


## 4.2 ACP Agent Registry

In previous topic, agents communicate via JSONL mailboxes and a coordinator
that knows the worker list at construction time. This is fine for a fixed
team, but breaks when agents are added or removed dynamically.

Hermes introduces **ACP (Agent Communication Protocol)**: a central registry
where agents **register themselves** at startup and **discover each other**
by capability rather than by name.

The coordinator does not hard-code `['investigator', 'tester']` --
it queries the registry: *'give me all agents with capability=code_review'*.

**Architecture:**
```
Agent A starts -> register(id='A', capabilities=['bash', 'git'])
Agent B starts -> register(id='B', capabilities=['code_review', 'testing'])

Coordinator -> discover(capability='code_review') -> [B]
           -> send_task(to='B', task='Review PR #42')
           -> B.mailbox receives task, executes, replies
```

**What this adds over the JSONL mailbox approach:**
- Dynamic team composition -- no hardcoded worker list
- Capability-based routing -- coordinator picks the right agent for each task
- Health-aware dispatch -- registry tracks last-seen and skips stale agents

**Sample Input / Output:**
```python
registry.register('agent_a', capabilities=['bash', 'file_ops'], endpoint='local')
registry.register('agent_b', capabilities=['code_review'],      endpoint='local')

registry.discover(capability='code_review')
# -> [AgentRecord(id='agent_b', capabilities=['code_review'], ...)]

registry.heartbeat('agent_a')
registry.get_active(max_age_s=30)
# -> ['agent_a', 'agent_b']  (both recently active)
```


In [ ]:
@dataclass
class AgentRecord:
    # One registered agent in the ACP registry.
    agent_id:     str
    capabilities: list  # list[str] -- what this agent can do
    endpoint:     str   # 'local' | 'ssh://host' | 'docker://container'
    registered_at: str = field(default_factory=lambda: datetime.now().isoformat())
    last_seen:    str  = field(default_factory=lambda: datetime.now().isoformat())
    metadata:     dict = field(default_factory=dict)


class ACPRegistry:
    # Central agent registry. Agents register on startup, deregister on shutdown.
    # Coordinator queries by capability to discover available workers dynamically.
    # Simplified version of Hermes acp_registry.

    def __init__(self, registry_path='.acp/registry.json'):
        self._path = Path(registry_path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        # _agents: dict[str, AgentRecord] (num_registered,)
        self._agents: dict = {}
        self._lock = threading.Lock()
        self._load()

    def _load(self):
        if self._path.exists():
            data = json.loads(self._path.read_text())
            for d in data:
                self._agents[d['agent_id']] = AgentRecord(**d)

    def _save(self):
        records = [r.__dict__ for r in self._agents.values()]
        self._path.write_text(json.dumps(records, indent=2))

    def register(self, agent_id, capabilities, endpoint='local', **metadata):
        # Register or re-register an agent.
        # agent_id: str, capabilities: list[str], endpoint: str -> AgentRecord
        with self._lock:
            record = AgentRecord(
                agent_id=agent_id, capabilities=capabilities,
                endpoint=endpoint, metadata=metadata,
            )
            self._agents[agent_id] = record
            self._save()
            print(f'  [ACP] Registered: {agent_id} caps={capabilities}')
            return record

    def deregister(self, agent_id):
        # agent_id: str -> None
        with self._lock:
            self._agents.pop(agent_id, None)
            self._save()
            print(f'  [ACP] Deregistered: {agent_id}')

    def heartbeat(self, agent_id):
        # Update last_seen timestamp. Agents call this periodically.
        # agent_id: str -> None
        with self._lock:
            if agent_id in self._agents:
                self._agents[agent_id].last_seen = datetime.now().isoformat()

    def discover(self, capability):
        # Find all registered agents that have a given capability.
        # capability: str -> list[AgentRecord] (num_matching,)
        with self._lock:
            return [
                r for r in self._agents.values()
                if capability in r.capabilities
            ]

    def get_active(self, max_age_s=60):
        # Return agents that have sent a heartbeat within max_age_s seconds.
        # max_age_s: int -> list[str] (agent_ids of active agents)
        cutoff = datetime.now().timestamp() - max_age_s
        with self._lock:
            active = []
            for r in self._agents.values():
                try:
                    ts = datetime.fromisoformat(r.last_seen).timestamp()
                    if ts >= cutoff:
                        active.append(r.agent_id)
                except Exception:
                    pass
            return active

    def route_task(self, task_description, required_capability):
        # Pick the best available agent for a task (first match, round-robin ready).
        # task_description: str, required_capability: str -> str | None (agent_id)
        candidates = self.discover(required_capability)
        active_ids = set(self.get_active())
        for c in candidates:
            if c.agent_id in active_ids:
                return c.agent_id
        # Fall back to any registered agent with the capability
        return candidates[0].agent_id if candidates else None

    def status_table(self):
        # -> str (human-readable registry state)
        with self._lock:
            if not self._agents:
                return 'Registry is empty.'
            lines = ['ACP Registry:']
            for r in self._agents.values():
                lines.append(f'  {r.agent_id:15} caps={r.capabilities} endpoint={r.endpoint}')
            return '\n'.join(lines)


# -- Demo -------------------------------------------------------------------
reg = ACPRegistry()

# Agents register themselves at startup
reg.register('bash_worker',   capabilities=['bash', 'file_ops', 'git'],   endpoint='local')
reg.register('review_worker', capabilities=['code_review', 'testing'],    endpoint='local')
reg.register('data_worker',   capabilities=['sql', 'data_analysis'],      endpoint='local')

# Simulate heartbeats
for aid in ['bash_worker', 'review_worker', 'data_worker']:
    reg.heartbeat(aid)

print(reg.status_table())
print(f'\nActive agents: {reg.get_active()}')
print(f'Discover code_review: {[r.agent_id for r in reg.discover("code_review")]}')
print(f'Route PR review task: {reg.route_task("Review PR #42", "code_review")}')

# Dynamic team -- no hardcoded list needed
coding_agents = reg.discover('bash')
print(f'\nAll bash-capable agents: {[r.agent_id for r in coding_agents]}')

## 4.3 Team Protocols

Raw one-shot delegation is efficient but risky for expensive or
irreversible tasks. The **plan-approval protocol** adds a review step:
the worker proposes a plan before executing, and the coordinator approves or
rejects it with feedback.

```
ASSIGNED
   |  worker proposes
   v
PLAN_PROPOSED --(coordinator rejects)--> PLAN_REJECTED
   |                                           |
   | (coordinator approves)          (worker revises)
   v                                           |
EXECUTING <----------------------------------------
   |
   v
DONE | FAILED
```

```
Worker:      'Plan: 1) Read tests  2) Add fixture  3) Add test case  4) Verify'
Coordinator: 'APPROVE'  ->  worker executes or  'REJECT: Missing error-path coverage'  ->  worker revises
```

In [ ]:
class TaskState(Enum):
    ASSIGNED      = auto()
    PLAN_PROPOSED = auto()
    PLAN_APPROVED = auto()
    PLAN_REJECTED = auto()
    EXECUTING     = auto()
    DONE          = auto()
    FAILED        = auto()


@dataclass
class CoordinatedTask:
    task_id:            str
    worker_id:          str
    description:        str
    state:              TaskState = TaskState.ASSIGNED
    proposed_plan:      Optional[str] = None
    rejection_feedback: Optional[str] = None
    result:             Optional[str] = None
    revision_count:     int = 0
    max_revisions:      int = 2


def _worker_propose(task):
    '''
    Worker generates a plan via a short LLM call.
    task: CoordinatedTask (state=ASSIGNED|PLAN_REJECTED) -> CoordinatedTask (PLAN_PROPOSED)
    '''
    prompt = task.description
    if task.rejection_feedback:
        prompt += f'\n\nPrev plan rejected. Feedback: {task.rejection_feedback}. Revise accordingly.'
    resp = client.chat.completions.create(
        model=MODEL,
        max_completion_tokens=200,
        messages=[
            {'role':'system','content':'You are a worker agent. Propose a concise plan (3-5 steps).'},
            {'role':'user','content': prompt},
        ],
    )
    task.proposed_plan = resp.choices[0].message.content
    task.state = TaskState.PLAN_PROPOSED
    print(f'  [Worker] Plan proposed: {task.proposed_plan[:60]}...')
    return task


def _coordinator_review(task):
    '''
    Coordinator reviews and approves or rejects the plan.
    task: CoordinatedTask (state=PLAN_PROPOSED) -> CoordinatedTask (PLAN_APPROVED|PLAN_REJECTED)
    '''
    resp = client.chat.completions.create(
        model=MODEL,
        max_completion_tokens=150,
        messages=[
            {'role':'system',
             'content':
                "You are a coordinator reviewing a worker's plan. "
                "Reply 'APPROVE' if sound. Reply 'REJECT: <feedback>' if it needs improvement."},
            {'role':'user','content': f'Task: {task.description}\n\nPlan:\n{task.proposed_plan}'},
        ],
    )
    verdict = resp.choices[0].message.content.strip()
    if verdict.upper().startswith('APPROVE'):
        task.state = TaskState.PLAN_APPROVED
        print(f'  [Coordinator] APPROVED')
    else:
        task.state = TaskState.PLAN_REJECTED
        task.rejection_feedback = verdict.replace('REJECT:','').strip()
        task.revision_count += 1
        print(f'  [Coordinator] REJECTED: {task.rejection_feedback[:60]}')
    return task


def run_plan_approval_protocol(description, worker_id='worker_a'):
    '''
    Run the full plan-approval FSM until approved or max revisions exceeded.
    description: str -> CoordinatedTask (final state)
    '''

    task = CoordinatedTask(
        task_id=f'ct_{uuid.uuid4().hex[:6]}',
        worker_id=worker_id,
        description=description,
    )
    while task.state not in {TaskState.PLAN_APPROVED, TaskState.DONE, TaskState.FAILED}:
        if task.revision_count > task.max_revisions:
            task.state = TaskState.FAILED
            print(f'  Max revisions ({task.max_revisions}) exceeded -> FAILED')
            break
        task = _worker_propose(task)
        task = _coordinator_review(task)
        if task.state == TaskState.PLAN_REJECTED:
            print(f'  Revision {task.revision_count}/{task.max_revisions}')
    return task

# -- Demo -------------------------------------------------------------------
print('Running plan-approval protocol...')
ct = run_plan_approval_protocol(
    'Refactor the auth module to use JWT instead of session cookies.'
)
print(f'\nFinal state: {ct.state.name}')
print(f'Approved plan:\n{ct.proposed_plan or "(none)"}')

Running plan-approval protocol...
  [Worker] Plan proposed: ...
  [Coordinator] REJECTED: 
  Revision 1/2
  [Worker] Plan proposed: ...
  [Coordinator] REJECTED: 
  Revision 2/2
  [Worker] Plan proposed: ...
  [Coordinator] REJECTED: 
  Revision 3/2
  Max revisions (2) exceeded -> FAILED

Final state: FAILED
Approved plan:
(none)


# 5) Autonomous Agents

Workers only act when explicitly delegated to.
This does not scale: 10 tasks require 10 manual coordinator assignments.

**Autonomous agents** self-direct by polling the shared task board,
claiming available tasks, executing them, and looping back.

```
spawn
  |
  v
WORK -- tool_calls -> LLM -> tool_calls --+
  |                                       | (loop)
  | finish_reason = stop                  |
  v                                       |
IDLE -- poll every 3s --------------------+
  |      \-- found unclaimed task? -> WORK
  |
  | no tasks for 30s
  v
SHUTDOWN
```

After context compaction, the summary may omit the agent identity.
Fix: the system prompt always states the agent ID and role.
It is never inside the compactable `messages[]` list.

Two workers may attempt to claim the same task simultaneously.
Resolution: the first to transition a task to `in_progress` wins.
The second sees it already `in_progress` and skips -- a simple optimistic lock.

## 5.1 Autonomous Worker

In [ ]:
class AutonomousWorker:
    # A self-directing worker that polls a shared FileTaskStore,
    # claims ready tasks, and executes them autonomously.

    POLL_INTERVAL_S = 3.0
    IDLE_TIMEOUT_S  = 30.0

    def __init__(self, agent_id, role, task_store, tools, tool_handlers):
        self.agent_id     = agent_id
        self.role         = role
        self.task_store   = task_store
        self.tools        = tools
        self.tool_handlers = tool_handlers
        self._stop        = threading.Event()

    @property
    def _system(self):
        # Identity re-injection: always in system prompt, never compactable.
        return (f"You are agent '{self.agent_id}', a {self.role}. "
                'Complete assigned tasks thoroughly and concisely.')

    def _try_claim(self):
        # Optimistic lock: first writer of in_progress wins.
        # -> Optional[Task] (claimed | None)
        for t in self.task_store.get_ready():
            if '[Error]' not in self.task_store.transition(t.id, 'in_progress'):
                claimed = self.task_store.get(t.id)
                if claimed and claimed.status == 'in_progress':
                    return claimed
        return None

    def _execute(self, task):
        # Fresh messages[] per task -- context isolation (section 4)
        # task: Task -> str (result summary)
        messages = [{'role':'user','content': f'Task: {task.description}'}]
        run_agent_loop(
            messages=messages, tools=self.tools, tool_handlers=self.tool_handlers,
            system=self._system, max_iterations=8,
        )
        return next((m['content'] for m in reversed(messages)
                     if m['role']=='assistant' and m['content']), '(no output)')

    def run(self):
        '''
        Main worker loop. Claim -> execute -> repeat.
        Designed to run in a daemon thread.
        '''

        print(f'[{self.agent_id}] Started.')
        idle_start = None
        while not self._stop.is_set():
            task = self._try_claim()
            if task:
                idle_start = None
                print(f'[{self.agent_id}] Claimed {task.id}: {task.description[:50]}')
                result = self._execute(task)
                self.task_store.transition(task.id, 'done')
                print(f'[{self.agent_id}] Done {task.id}: {result[:80]}')
            else:
                if idle_start is None:
                    idle_start = time.time()
                    print(f'[{self.agent_id}] Idle. Polling every {self.POLL_INTERVAL_S}s...')
                if time.time() - idle_start >= self.IDLE_TIMEOUT_S:
                    print(f'[{self.agent_id}] Idle timeout. Shutting down.')
                    break
                time.sleep(self.POLL_INTERVAL_S)
        print(f'[{self.agent_id}] Stopped.')

    def shutdown(self): self._stop.set()

    def start(self):
        t = threading.Thread(target=self.run, daemon=True)
        t.start()
        return t


# === Demo ===
auto_store = FileTaskStore('.tasks/auto.jsonl')
auto_store._tasks.clear(); auto_store._flush()
auto_store.create(TaskSpec("Echo 'hello from worker' via bash"))
auto_store.create(TaskSpec('Print current date via bash'))
print('Task board:'); print(auto_store.summary())

worker = AutonomousWorker(
    agent_id='alpha', role='general-purpose bash executor',
    task_store=auto_store, tools=CORE_TOOL_SCHEMAS, tool_handlers=CORE_TOOL_HANDLERS,
)
worker.IDLE_TIMEOUT_S = 10.0
t = worker.start()
t.join(timeout=25)
print('\nFinal task board:'); print(auto_store.summary())

Task board:
-- [task_1] Echo 'hello from worker' via bash | deps: none
-- [task_2] Print current date via bash | deps: none
[alpha] Started.
[alpha] Claimed task_1: Echo 'hello from worker' via bash
[alpha] Done task_1: hello from worker
[alpha] Claimed task_2: Print current date via bash
[alpha] Done task_2: Tue Apr 21 09:35:23 AM UTC 2026
[alpha] Idle. Polling every 3.0s...
[alpha] Idle timeout. Shutting down.
[alpha] Stopped.

Final task board:
OK [task_1] Echo 'hello from worker' via bash | deps: none
OK [task_2] Print current date via bash | deps: none


## 5.2 Worktree Isolation

When multiple autonomous agents work simultaneously, they may edit the same
files -- causing race conditions, conflicting writes, and corrupted state.
**Worktree isolation** gives each task a private working directory.

Using `git worktree`, each task gets its own branch and directory.
Workers commit in their lane; the coordinator merges when done.

```
create(task_id) -> new branch + isolated directory
worker writes / commits inside worktree_path/
commit(task_id, message) -> git commit inside worktree
merge(task_id) -> coordinator merges into main (manual step)
remove(task_id) -> cleanup branch + directory
```

**Scoped handlers**: the harness wraps `file_read`, `file_write`, and `bash`
to prepend the worktree path. Workers call tools with relative paths;
the harness transparently scopes them. No worker can accidentally write
to the main repository.

**Fallback**: in non-git environments (Colab without a repo), the manager
falls back to `tempfile.mkdtemp()` -- same API, no VCS.

In [ ]:
@dataclass
class Worktree:
    task_id: str
    path:    Path
    branch:  Optional[str]  # None if not using git
    is_git:  bool


class WorktreeManager:
    '''
    Manages isolated working directories for concurrent tasks.
    Prefers git worktrees; falls back to tmpdir in non-git environments.
    '''
    def __init__(self, base_dir='.worktrees'):
        self.base_dir  = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)
        # _worktrees: dict[str, Worktree] (num_active,)
        self._worktrees = {}
        self._is_git    = self._detect_git()

    @staticmethod
    def _detect_git():
        r = subprocess.run('git rev-parse --git-dir', shell=True, capture_output=True)
        return r.returncode == 0

    def create(self, task_id):
        # task_id: str -> Worktree
        if task_id in self._worktrees: return self._worktrees[task_id]
        return self._git_worktree(task_id) if self._is_git else self._tmpdir_worktree(task_id)

    def _git_worktree(self, task_id):
        branch = f'harness/{task_id}'
        path   = self.base_dir / task_id
        r = subprocess.run(
            f'git worktree add -b {branch} {path}',
            shell=True, capture_output=True, text=True
        )
        if r.returncode != 0:  # branch already exists
            subprocess.run(
                f'git worktree add {path} {branch}',
                shell=True,
                capture_output=True
            )
        wt = Worktree(task_id=task_id, path=path, branch=branch, is_git=True)
        self._worktrees[task_id] = wt
        print(f'  [Worktree] git: {path} (branch: {branch})')
        return wt

    def _tmpdir_worktree(self, task_id):
        path = Path(tempfile.mkdtemp(prefix=f'wt_{task_id}_'))
        wt   = Worktree(task_id=task_id, path=path, branch=None, is_git=False)
        self._worktrees[task_id] = wt
        print(f'  [Worktree] tmpdir: {path}')
        return wt

    def get_path(self, task_id):
        wt = self._worktrees.get(task_id)
        return wt.path if wt else None

    def commit(self, task_id, message):
        # task_id: str, message: str -> str (git output | info)
        wt = self._worktrees.get(task_id)
        if not wt or not wt.is_git: return '[Info] Not a git worktree -- skipping.'
        r = subprocess.run(f'cd {wt.path} && git add -A && git commit -m "{message}"',
                           shell=True, capture_output=True, text=True)
        return (r.stdout + r.stderr).strip()

    def remove(self, task_id):
        # task_id: str -> str (confirmation | error)
        wt = self._worktrees.pop(task_id, None)
        if not wt: return f'[Error] No worktree for: {task_id}'
        if wt.is_git:
            subprocess.run(f'git worktree remove --force {wt.path}',
                           shell=True, capture_output=True)
            subprocess.run(f'git branch -D {wt.branch}',
                           shell=True, capture_output=True)
        else:
            shutil.rmtree(wt.path, ignore_errors=True)
        return f'Removed worktree for {task_id}'

    def build_scoped_handlers(self, task_id, base_handlers):
        # Wrap file and bash handlers to scope all paths to the task worktree.
        # task_id: str, base_handlers: dict[str, Callable] (N,)
        # -> dict[str, Callable] (N,) -- scoped handlers
        wt_path = self.get_path(task_id)
        if not wt_path: return base_handlers

        def _scope(p): return str(wt_path / p) if not Path(p).is_absolute() else p

        return {
            **base_handlers,
            'file_read':  lambda fp, **kw: base_handlers['file_read'](_scope(fp), **kw),
            'file_write': lambda fp, c:    base_handlers['file_write'](_scope(fp), c),
            'bash':       lambda cmd, **kw: base_handlers['bash'](f'cd {wt_path} && {cmd}', **kw),
        }


# -- Demo -------------------------------------------------------------------
wm   = WorktreeManager()
wt_a = wm.create('task_alpha')
wt_b = wm.create('task_beta')
print(f'Worktree A: {wt_a.path} (git={wt_a.is_git})')
print(f'Worktree B: {wt_b.path} (git={wt_b.is_git})')
(wt_a.path / 'output_a.txt').write_text('Worker A was here')
print(f'output_a in A: {(wt_a.path / "output_a.txt").exists()}')
print(f'output_a in B (expect False): {(wt_b.path / "output_a.txt").exists()}')
wm.remove('task_alpha'); wm.remove('task_beta')
print('Cleaned up.')

  [Worktree] tmpdir: /tmp/wt_task_alpha_vgbiagb7
  [Worktree] tmpdir: /tmp/wt_task_beta_d0x8fdfd
Worktree A: /tmp/wt_task_alpha_vgbiagb7 (git=False)
Worktree B: /tmp/wt_task_beta_d0x8fdfd (git=False)
output_a in A: True
output_a in B (expect False): False
Cleaned up.


# 6) Agent Harness

`Agent Harness` combines every component into a single class.

```
run(user_message)
   |
   |-- _pre_turn()
   |    |-- drain background notifications -> inject as user msg   
   |    \-- compact_messages() if over token budget               
   |
   |-- run_agent_loop()                                           
   |    |-- bash / file_read / file_write / glob / grep            
   |    |-- todo (write / update / read)                           
   |    |-- agent (subagent spawning)                              
   |    |-- list_skills / load_skill                               
   |    \-- run_background                                         
   |         \-- on_tool_call: PermissionGate + nag reminder
   |
   \-- returns final assistant text
```

In [ ]:
@dataclass
class PermissionGate:
    # Blocks dangerous commands and logs every tool call for auditing.
    deny_prefixes: list = field(default_factory=lambda: [
        'rm -rf /', 'sudo rm -rf', 'mkfs', '> /dev/sd',
    ])
    audit_log: list = field(default_factory=list)

    def check(self, tool_name, tool_input):
        # (tool_name, tool_input) -> (allowed: bool, reason: str)
        if tool_name == 'bash':
            cmd = tool_input.get('command', '')
            for prefix in self.deny_prefixes:
                if cmd.strip().startswith(prefix):
                    return False, f'Blocked: matches deny prefix "{prefix}"'
        return True, 'Allowed'

    def gate(self, tool_name, tool_input):
        # Log and raise RuntimeError if blocked.
        allowed, reason = self.check(tool_name, tool_input)
        self.audit_log.append({'time': datetime.now().isoformat(),
                                'tool': tool_name, 'allowed': allowed})
        if not allowed:
            raise RuntimeError(f'[PermissionGate] {reason}')


class AgentHarness:
    # Full agent harness combining sections 1-12.
    # Single entry point: run(user_message) handles everything automatically.

    def __init__(self, session_id=None):
        self.session_id = session_id or uuid.uuid4().hex[:8]
        # messages: list[dict] -- grows across run() calls (multi-turn state)
        self.messages   = []

        # -- Subsystems -----------------------------------------------------
        self.todo       = TodoManager()
        self.notify_q   = NotificationQueue()
        self.bg_runner  = BackgroundTaskRunner(self.notify_q)
        self.permission = PermissionGate()
        self.skill_lib  = SkillLibrary()

        skill_schemas, skill_handlers = build_skill_tools(self.skill_lib)

        # -- System prompt (outside messages[], survives compaction) ----------
        self.system = (
            'You are an expert software engineering agent. '
            'Before complex tasks ALWAYS call todo(action=write) to plan. '
            'Use list_skills / load_skill for domain knowledge on demand. '
            'Use run_background for slow commands so you can keep working.\n\n'
            f'Session: {self.session_id}'
        )

        # -- Tool registry --------------------------------------------------
        bg_schema = {
            'type':'function','function':{
                'name':'run_background',
                'description':
                    'Run a slow shell command in the background. Returns immediately; '
                    'result arrives via notification before the next LLM call.',
                'parameters':{'type':'object',
                    'properties':{'command':{'type':'string'},
                                  'task_id':{'type':'string'}},
                    'required':['command'],
                },
            },
        }
        self.tools = (
            CORE_TOOL_SCHEMAS + [TODO_TOOL_SCHEMA, AGENT_TOOL_SCHEMA]
            + skill_schemas + [bg_schema]
        )
        agent_handler = build_agent_tool(
            parent_system=self.system,
            tools=CORE_TOOL_SCHEMAS + skill_schemas,
            tool_handlers={**CORE_TOOL_HANDLERS, **skill_handlers},
        )
        todo_handler = build_todo_handler(self.todo)
        self.handlers = {
            **CORE_TOOL_HANDLERS, **skill_handlers,
            'todo': todo_handler, 'agent': agent_handler,
            'run_background': self.bg_runner.run,
        }
        self._call_count = 0

    def _pre_turn(self):
        '''
        # Housekeeping before each LLM call:
        # 1. Drain background notifications -> inject as user message.
        # 2. Compact messages[] if approaching token budget.
        '''

        notifs = self.notify_q.format_and_drain()
        if notifs:
            self.messages.append({'role':'user','content': notifs})
        self.messages = compact_messages(
            self.messages, system=self.system,
            token_budget=60_000, trigger_ratio=0.75,
        )

    def _on_tool_call(self, tool_name, tool_input):
        # Permission gate + nag reminder on every tool call.
        self._call_count += 1
        # raises on block
        self.permission.gate(tool_name, tool_input)
        nag = self.todo.get_nag_reminder(self._call_count)
        if nag:
            print(f'\n  [Nag] {nag}\n')
        print(f'  [{tool_name}] {str(tool_input)[:100]}')

    def run(self, user_message):
        # Process one user message and return the agent text reply.
        # user_message: str -> str (agent final reply)
        self._pre_turn()
        self.messages.append({'role':'user','content': user_message})
        run_agent_loop(
            messages=self.messages, tools=self.tools, tool_handlers=self.handlers,
            system=self.system, on_tool_call=self._on_tool_call,
        )
        return next(
            (m['content'] for m in reversed(self.messages)
             if m['role']=='assistant' and m['content']),
            '(no reply)',
        )

    def save_session(self, out_dir='.sessions'):
        # Persist the full conversation to disk as JSON.
        Path(out_dir).mkdir(exist_ok=True)
        p = Path(out_dir) / f'{self.session_id}.json'
        p.write_text(json.dumps(self.messages, indent=2, default=str), encoding='utf-8')
        return str(p)


# -- Capstone Demo ----------------------------------------------------------
print('Initialising AgentHarness...')
harness = AgentHarness()
print(f'Session: {harness.session_id}')
print(f'Tools:   {[t["function"]["name"] for t in harness.tools]}\n')

print('='*60)
print('Turn 1')
print('='*60)
reply1 = harness.run(
    'What Python version is installed? List files in the current directory.'
)
print(f'\nAgent:\n{reply1}\n')

print('='*60)
print('Turn 2')
print('='*60)
reply2 = harness.run(
    "Create hello.py that prints 'Hello from the harness!', then run it."
)
print(f'\nAgent:\n{reply2}\n')

saved = harness.save_session()
print(f'Session saved: {saved}')
print(f'Total tool calls:    {harness._call_count}')
print(f'Audit log entries:   {len(harness.permission.audit_log)}')

Initialising AgentHarness...
Session: e77c8566
Tools:   ['bash', 'file_read', 'file_write', 'glob', 'grep', 'todo', 'agent', 'list_skills', 'load_skill', 'run_background']

Turn 1
  [todo] {'action': 'write', 'tasks': [{'description': 'Determine installed Python version by running python 
  [bash] {'command': 'python --version && python3 --version || true', 'timeout': 120000}
  [bash] {'command': 'ls -la', 'timeout': 120000}
  [todo] {'action': 'update', 'todo_id': 'todo_1', 'status': 'done'}

  [Nag] [Reminder] 1 task(s) unfinished. Update status as you complete them.
OK [todo_1] Determine installed Python version by running python --version (or python3 --version if needed). (done)
-- [todo_2] List files in the current directory (ls -la). (pending)

  [todo] {'action': 'update', 'todo_id': 'todo_2', 'status': 'done'}

Agent:
- Python version: Python 3.12.13
- Files in current directory (ls -la):
  total 28
  drwxr-xr-x 1 root root 4096 Apr 21 09:35 .
  drwxr-xr-x 1 root root 4096 Apr 